# Visualización de las métricas obtenidas en el entrenamiento

In [2]:
import json
import argparse #para obterner los argumentos desde consola
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

Función para cargar el JSON y transformar la columna *`log_history`* a DataFrame de Pandas

In [3]:
def load_and_process_data(filepath):
    """
    Carga el JSON y convierte el log_history en un DataFrame de Pandas.
    Separa métricas de entrenamiento y evaluación.
    """
    try:
        with open(filepath, 'r') as f:
            data = json.load(f)
        
        # Extraemos el historial
        if 'log_history' not in data:
            raise ValueError("El archivo JSON no contiene 'log_history'.")
            
        df = pd.DataFrame(data['log_history'])
        
        print(df.head())
        # Los logs de HuggingFace suelen tener filas separadas para train y eval.
        
        return df
    except Exception as e:
        print(f"Error cargando el archivo: {e}")
        return None

Función para la creación de gráficas.

In [4]:
def plot_training_results(df: pd.DataFrame, output_dir: str, model_alias:str,grafLoss:bool =True,grafLR:bool=True,grafAcc:bool=True)->None:
    """
    Genera gráficas eficientemente usando Seaborn y guarda los resultados.
    
    Args:

        - df (pd.DataFrame):Datos de 'log_history' del JSON
        - output_dir (str): Ruta del directorio donde se guardan las gráficas
        - model_alias (str): alias con el que se conoce al modelo entrenado 
        - grafLoss (bool): Whether to plot the loss curves. Default to True.
        - grafLR (bool): Whether to plot the learning rate evolution along steps. Default to True.
        - grafAcc (bool): Whether to plot extra training and evaluation metrics. Default to True.
   
    """
    # Configuración de estilo
    sns.set_theme(style="whitegrid")
    
    # Crear carpeta de salida si no existe
    os.makedirs(output_dir, exist_ok=True)
    

    # Filtramos DataFrames para entrenamiento y evaluación
    # Usamos 'loss' para detectar filas de train y 'eval_loss' para filas de eval
    train_df = df[df['loss'].notna()].copy()
    eval_df = df[df['eval_loss'].notna()].copy()
    
    # ---------------------------------------------------------
    # GRAFICA 1: Loss (Pérdida) - Train vs Eval
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 6))
    
    if not train_df.empty:
        sns.lineplot(data=train_df, x='step', y='loss', label='Train Loss', color='blue', alpha=0.6)
    
    if not eval_df.empty:
        # A veces eval tiene menos puntos, seaborn maneja la interpolación visualmente
        sns.lineplot(data=eval_df, x='step', y='eval_loss', label='Eval Loss', color='red', marker='o')
        
    plt.title(f'Curvas de Pérdida (Loss)\n{model_alias}')
    plt.xlabel('Global Step')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(os.path.join(output_dir, f"{model_alias}_loss.png"), dpi=300)
    plt.close() # Cierra la figura para liberar memoria
    print(f"-> Gráfica guardada: {model_alias}_loss.png")

    # ---------------------------------------------------------
    # GRAFICA 2: Learning Rate y Epoch
    # ---------------------------------------------------------
    if 'learning_rate' in train_df.columns:
        fig, ax1 = plt.subplots(figsize=(10, 6))

        color = 'tab:green'
        ax1.set_xlabel('Step')
        ax1.set_ylabel('Learning Rate', color=color)
        sns.lineplot(data=train_df, x='step', y='learning_rate', ax=ax1, color=color)
        ax1.tick_params(axis='y', labelcolor=color)

        # Segundo eje para ver el progreso de épocas
        ax2 = ax1.twinx()  
        color = 'tab:gray'
        ax2.set_ylabel('Epoch', color=color)
        sns.lineplot(data=df, x='step', y='epoch', ax=ax2, color=color, linestyle='--')
        ax2.tick_params(axis='y', labelcolor=color)

        plt.title(f'Learning Rate Schedule & Epochs\n{model_alias}')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"{model_alias}_lr.png"), dpi=300)
        plt.close()
        print(f"-> Gráfica guardada: {model_alias}_lr.png")

    # ---------------------------------------------------------
    # GRAFICA 3: Métricas adicionales (Accuracy, Entropy)
    # ---------------------------------------------------------
    # Detectamos columnas dinámicamente que contengan "accuracy" o métricas similares
    potential_metrics = [col for col in df.columns if 'accuracy' in col or 'entropy' in col]
    
    # Agrupamos métricas de train y eval que sean pares (ej: mean_token_accuracy vs eval_mean_token_accuracy)
    base_metrics = set()
    for m in potential_metrics:
        if m.startswith('eval_'):
            base_metrics.add(m.replace('eval_', ''))
        else:
            base_metrics.add(m)

    for metric in base_metrics:
        plt.figure(figsize=(10, 6))
        has_data = False
        
        # Plot Train
        if metric in train_df.columns:
            sns.lineplot(data=train_df, x='step', y=metric, label=f'Train {metric}', alpha=0.6)
            has_data = True
            
        # Plot Eval
        eval_metric = f"eval_{metric}"
        if eval_metric in eval_df.columns:
            sns.lineplot(data=eval_df, x='step', y=eval_metric, label=f'Eval {metric}', marker='o')
            has_data = True
            
        if has_data:
            plt.title(f'Métrica: {metric}\n{model_alias}')
            plt.xlabel('Step')
            plt.ylabel(metric)
            plt.legend()
            plt.savefig(os.path.join(output_dir, f"{model_alias}_{metric}.png"), dpi=300)
            plt.close()
            print(f"-> Gráfica guardada: {model_alias}_{metric}.png")

Función main. Está diseñada para que se pueda llamar desde consolo al método. Es posible pasarle tantos JSONs como se quiera.

In [5]:
def main(model_alias:str):
    
    # Inicializamos el analizador de argumentos para interactuar con la terminal
    parser = argparse.ArgumentParser(description="Procesar logs de entrenamiento (JSON) y generar gráficas.")
    
    # Argumento posicional: permite pasar uno o más archivos (gracias a nargs='+')
    # Ejemplo: python script.py file1.json file2.json
    #metavars='F' indica que se esperan ficheros, no carpetas
    parser.add_argument('files', metavar='F', type=str, nargs='+', help='Ruta al archivo(s) .json') #interesante el nargs
    
    # Argumento opcional: permite definir la ruta de salida donde se guardarán las gráficas generadas
    # Tiene valor por defecto
    parser.add_argument('--out', type=str, default=f'./{model_alias}/plots', help='Directorio de salida para las gráficas')

    # Parseamos los argumentos introducidos por el usuario
    args = parser.parse_args()

    # Procesamiento por lotes: iteramos sobre cada archivo proporcionado
    for file_path in args.files:
        # Validación de seguridad: verificamos que el archivo realmente exista en el disco
        if os.path.exists(file_path):
            print(f"--- Processing: {file_path} ---")
            
            # Carga de datos mediante la función auxiliar
            df = load_and_process_data(file_path)
            
            if df is not None:
                # Extracción del nombre base del archivo para nombrar las gráficas
                # os.path.basename: quita la ruta (ej: 'logs/data.json' -> 'data.json')
                # os.path.splitext: quita la extensión (ej: 'data.json' -> 'data')
                
                
                # Ejecución de la lógica de graficado
                plot_training_results(df, args.out, model_alias=model_alias)
        else:
            # Notificación de error amable si una de las rutas es incorrecta
            print(f"Error: File not found at {file_path}")



In [7]:
if __name__ == "__main__":
    model_alias="MedTrinity25M_full"
    if model_alias=="MedTrinity25M_full": # MedTrinity25M_full se divide en subdirectorios por su tamaño. 
        #Debería tambien hacerse en la versión demo si se vovliese a modificar
        model_size="10k"
        sys.argv = ['plot_trainer.py', f'./{model_alias}/{model_alias}_{model_size}/trainer_state.json', '--out', f'./{model_alias}/{model_alias}_{model_size}/plots']
    else:
        sys.argv = ['plot_trainer.py', f'./{model_alias}/trainer_state.json', '--out', f'./{model_alias}/plots']
    main(model_alias)

--- Processing: ./MedTrinity25M_full/MedTrinity25M_full_10k/trainer_state.json ---
    epoch  eval_entropy  eval_loss  eval_mean_token_accuracy  eval_num_tokens  \
0  0.1920      0.331599   0.523643                  0.828436         844016.0   
1  0.3072           NaN        NaN                       NaN              NaN   
2  0.3840      0.188944   0.362817                  0.874772        1692475.0   
3  0.5760      0.165986   0.320509                  0.886190        2538394.0   
4  0.6144           NaN        NaN                       NaN              NaN   

   eval_runtime  eval_samples_per_second  eval_steps_per_second  step  \
0       20.3830                    2.944                  0.736    30   
1           NaN                      NaN                    NaN    48   
2       20.7470                    2.892                  0.723    60   
3       20.5929                    2.914                  0.728    90   
4           NaN                      NaN                    NaN  